# 04 · VLM success detector: calibrate on real frames

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/vla_ros2/blob/main/notebooks/04_vlm_success.ipynb)

`vla_success_detector_node` asks a VLM "has the robot completed: <instruction>?" and
thresholds P(yes). This notebook picks that threshold on real camera frames: the first
frame of each `svla_so100_stacking` episode is labeled *not done*, the last *done*.
AUROC 0.5 means the VLM cannot tell them apart.

Measured on CPU (2026-10-11), episodes 0–2 (3 done / 3 not-done frames, so noisy):

| VLM | camera | prompt | AUROC |
|---|---|---|---:|
| SmolVLM2-500M | top | default | 0.11 |
| SmolVLM2-500M | wrist | default | 0.22 |
| SmolVLM2-500M | top | goal-state | 0.11 |
| SmolVLM2-500M | top | cube relation | 0.67 |
| SmolVLM2-2.2B | top | default | 0.56 |

None separates done from not-done reliably, although 500M passes the synthetic
cartoon test. From the top camera a stacked cube is a red square with a thin blue edge.
Re-run on a GPU with more episodes before trusting any threshold.

In [ ]:
# Colab: clone the repo. Locally: run from the repo (or its notebooks/ folder).
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("vla_ros2").exists():
    !git clone --depth 1 https://github.com/rsasaki0109/vla_ros2.git
REPO = Path("vla_ros2").resolve() if IN_COLAB else Path.cwd()
if not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
%pip install -q -e ".[smolvla,vlm]"

In [ ]:
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Sanity check on synthetic scenes

In [ ]:
!{sys.executable} scripts/vlm_success_check.py --device {DEVICE} 2>&1 | grep -v -i warn

## Calibrate on real frames

In [ ]:
from vla_ros2.vlm import DEFAULT_PROMPT_TEMPLATE as DEFAULT

RELATION = 'Task: "{instruction}" Is the red cube sitting on top of the blue cube right now? Answer yes or no.'
TOP, WRIST = "observation.images.top", "observation.images.wrist"
CANDIDATES = [  # (model, camera, prompt template)
    ("HuggingFaceTB/SmolVLM2-500M-Video-Instruct", TOP, DEFAULT),
    ("HuggingFaceTB/SmolVLM2-500M-Video-Instruct", TOP, RELATION),
    ("HuggingFaceTB/SmolVLM2-2.2B-Instruct", TOP, DEFAULT),
    ("HuggingFaceTB/SmolVLM2-2.2B-Instruct", TOP, RELATION),
    ("HuggingFaceTB/SmolVLM2-2.2B-Instruct", WRIST, RELATION),
]
EPISODES = list(range(10))

In [ ]:
import json
import subprocess


def run(cmd):
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[-3000:])
    return proc.stdout


calibrations = []
for pretrained, camera, prompt in CANDIDATES:
    out = f"/tmp/vlm_cal_{len(calibrations)}.json"
    cmd = [
        sys.executable, "scripts/vlm_calibrate_lerobot.py", "--device", DEVICE,
        "--pretrained", pretrained, "--camera", camera,
        *([] if prompt == DEFAULT else ["--prompt-template", prompt]),
        "--episodes", *map(str, EPISODES), "--out", out,
    ]
    print(run(cmd).strip().splitlines()[-1])
    calibrations.append(json.load(open(out)))

In [ ]:
for c in calibrations:
    prompt = "default" if c["prompt_template"] == DEFAULT else "relation"
    print(f"{c['model_id']:<44} {c['camera']:<26} {prompt:<9} AUROC={c['auroc']:.2f} "
          f"threshold={c['threshold']:.3f} balanced_acc={c['balanced_accuracy']:.2f}")

Use the best row only if its AUROC is clearly above 0.5, then launch with it:

```bash
ros2 launch vla_ros2 success_detector.launch.py \
  pretrained:=HuggingFaceTB/SmolVLM2-2.2B-Instruct
# and set threshold / image_topic in config/success_detector.yaml
```